# Anchors and key transitions

Sign and verify checkpoints and key handovers on all three wire majors, and
refuse altered checkpoint context.

Install Deno and register its TypeScript kernel with `deno jupyter --install`.
Open this file in a Jupyter front end such as JupyterLab or VS Code, select
**Deno**, then Run All in a fresh kernel. The cells import exact published
signer **2.0.1** and verifier **2.0.1**.

Fixed times describe demonstration credentials. Generated keys, signatures and
nonces stay in memory. Signing and verification do not grant operational
authority; the host owns trust, live revocation, replay and policy.

## Load the published packages

Check both release identities and the backend Deno selects.


In [ ]:
import assert from "node:assert/strict";
import * as signer from "npm:@bounded-authority-protocol/signer@2.0.1";
import * as verifier from "npm:@bounded-authority-protocol/verifier@2.0.1";
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.1/package.json" with {
  type: "json",
};
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.1/package.json" with {
  type: "json",
};
assert.equal(signerMetadata.version, "2.0.1");
assert.equal(verifierMetadata.version, "2.0.1");
assert.equal(verifier.BACKEND_ID, "portable");
console.log(
  JSON.stringify({
    signer: signerMetadata.version,
    verifier: verifierMetadata.version,
    backend: verifier.BACKEND_ID,
  }),
);

{"signer":"2.0.1","verifier":"2.0.1","backend":"portable"}


## Prepare real demonstration key handles

Each constructor generates a fresh non-extractable WebCrypto key pair and wraps
it with the signer's own handle factory, so the private key never leaves
WebCrypto. No key material is displayed. Production custody supplies this same
key-handle contract.

In [ ]:
async function createEd25519Handle(
  keyId: string,
  role: signer.KeyRole = "holder",
): Promise<signer.KeyHandle> {
  const pair = await crypto.subtle.generateKey({ name: "Ed25519" }, false, [
    "sign",
    "verify",
  ]) as CryptoKeyPair;
  const made = await signer.ed25519KeyHandle(pair, { keyId, role });
  if (!made.ok) throw new Error(made.error);
  return made.value;
}
async function createEs256Handle(
  keyId: string,
  role: signer.KeyRole = "holder",
): Promise<signer.Es256KeyHandle> {
  const pair = await crypto.subtle.generateKey(
    { name: "ECDSA", namedCurve: "P-256" },
    false,
    ["sign", "verify"],
  ) as CryptoKeyPair;
  const made = await signer.es256KeyHandle(pair, { keyId, role });
  if (!made.ok) throw new Error(made.error);
  return made.value;
}
const ed25519Width =
  (await (await createEd25519Handle("example-ed25519")).publicKey()).length;
const es256Width =
  (await (await createEs256Handle("example-es256")).publicKey()).length;
assert.equal(ed25519Width, 32);
assert.equal(es256Width, 65);
console.log(
  JSON.stringify({ ed25519Bytes: ed25519Width, es256Bytes: es256Width }),
);

{"ed25519Bytes":32,"es256Bytes":65}


## Sign a major 1 checkpoint

The current key signs the chain hash at a known sequence.


In [ ]:
const current1 = await createEd25519Handle("current-1");
const next1 = await createEd25519Handle("next-1");
const currentKey1 = {
  keyId: "current-1",
  publicKey: await current1.publicKey(),
  validFrom: 1_000,
  validBefore: 2_000,
};
const nextKey1 = {
  keyId: "next-1",
  publicKey: await next1.publicKey(),
  validFrom: 1_500,
  validBefore: null,
};
const currentFingerprint1 = verifier.publicKeyThumbprintRaw(
  currentKey1.publicKey,
);
const nextFingerprint1 = verifier.publicKeyThumbprintRaw(nextKey1.publicKey);
const checkpoint1 = {
  anchorId: "anchor-1",
  chainId: "audit:approvals",
  sequence: 0,
  chainHash: new Uint8Array(32),
};
const anchor1 = await signer.signAnchor(checkpoint1, current1, {
  anchoredAt: 1_400,
});
assert.ok(anchor1.ok);
console.log(JSON.stringify({ ok: anchor1.ok }));

{"ok":true}


## Verify the major 1 checkpoint

The caller supplies trusted key windows and the exact checkpoint fingerprint.


In [ ]:
const expectedAnchor1 = {
  ...checkpoint1,
  keyId: "current-1",
  anchoredAt: 1_400,
  keyFingerprint: currentFingerprint1,
};
const anchorFacts1 = verifier.verifyHistoricalAnchor(
  anchor1.value.anchor,
  currentKey1,
  expectedAnchor1,
);
assert.ok(anchorFacts1.ok);
console.log(
  JSON.stringify({
    ok: anchorFacts1.ok,
    verification: anchorFacts1.value.verification,
    trust: anchorFacts1.value.trust,
  }),
);

{"ok":true,"verification":"signature_and_window","trust":"not_evaluated"}


## Refuse a changed major 1 checkpoint

A different chain hash must not verify under the original compact.


In [ ]:
const wrongHash1 = verifier.verifyHistoricalAnchor(
  anchor1.value.anchor,
  currentKey1,
  { ...expectedAnchor1, chainHash: new Uint8Array(32).fill(1) },
);
assert.deepEqual(wrongHash1, { ok: false });
console.log(JSON.stringify(wrongHash1));

{"ok":false}


## Sign a major 1 key transition

Only the current key signs the handover to the next public key.


In [ ]:
const transitionInput1 = {
  transitionId: "transition-1",
  chainId: "audit:approvals",
  effectiveAt: 1_500,
  nextKeyId: "next-1",
  nextPublicKey: nextKey1.publicKey,
};
const transition1 = await signer.signKeyTransition(transitionInput1, current1);
assert.ok(transition1.ok);
console.log(JSON.stringify({ ok: transition1.ok }));

{"ok":true}


## Verify the major 1 handover

Expected key identifiers and fingerprints come from the caller, independent of
the compact.

In [ ]:
const transitionFacts1 = verifier.verifyKeyTransition(
  transition1.value.keyTransition,
  currentKey1,
  nextKey1,
  {
    transitionId: "transition-1",
    chainId: "audit:approvals",
    effectiveAt: 1_500,
    currentKeyId: "current-1",
    nextKeyId: "next-1",
    currentKeyFingerprint: currentFingerprint1,
    nextKeyFingerprint: nextFingerprint1,
  },
);
assert.ok(transitionFacts1.ok);
console.log(
  JSON.stringify({
    ok: transitionFacts1.ok,
    verification: transitionFacts1.value.verification,
  }),
);

{"ok":true,"verification":"authenticated_transition"}


## Sign a major 2 checkpoint

The current key signs the chain hash at a known sequence.


In [ ]:
const current2 = await createEd25519Handle("current-2");
const next2 = await createEd25519Handle("next-2");
const currentKey2 = {
  keyId: "current-2",
  publicKey: await current2.publicKey(),
  validFrom: 1_000,
  validBefore: 2_000,
};
const nextKey2 = {
  keyId: "next-2",
  publicKey: await next2.publicKey(),
  validFrom: 1_500,
  validBefore: null,
};
const currentFingerprint2 = verifier.publicKeyThumbprintRaw(
  currentKey2.publicKey,
);
const nextFingerprint2 = verifier.publicKeyThumbprintRaw(nextKey2.publicKey);
const checkpoint2 = {
  anchorId: "anchor-2",
  chainId: "audit:approvals",
  sequence: 0,
  chainHash: new Uint8Array(32),
};
const anchor2 = await signer.signV2Anchor(checkpoint2, current2, {
  anchoredAt: 1_400,
});
assert.ok(anchor2.ok);
console.log(JSON.stringify({ ok: anchor2.ok }));

{"ok":true}


## Verify the major 2 checkpoint

The caller supplies trusted key windows and the exact checkpoint fingerprint.


In [ ]:
const expectedAnchor2 = {
  ...checkpoint2,
  keyId: "current-2",
  anchoredAt: 1_400,
  keyFingerprint: currentFingerprint2,
};
const anchorFacts2 = verifier.v2.verifyHistoricalAnchor(
  anchor2.value.anchor,
  currentKey2,
  expectedAnchor2,
);
assert.ok(anchorFacts2.ok);
console.log(
  JSON.stringify({
    ok: anchorFacts2.ok,
    verification: anchorFacts2.value.verification,
    trust: anchorFacts2.value.trust,
  }),
);

{"ok":true,"verification":"signature_and_window","trust":"not_evaluated"}


## Refuse a changed major 2 checkpoint

A different chain hash must not verify under the original compact.


In [ ]:
const wrongHash2 = verifier.v2.verifyHistoricalAnchor(
  anchor2.value.anchor,
  currentKey2,
  { ...expectedAnchor2, chainHash: new Uint8Array(32).fill(1) },
);
assert.deepEqual(wrongHash2, { ok: false });
console.log(JSON.stringify(wrongHash2));

{"ok":false}


## Sign a major 2 key transition

Only the current key signs the handover to the next public key.


In [ ]:
const transitionInput2 = {
  transitionId: "transition-2",
  chainId: "audit:approvals",
  effectiveAt: 1_500,
  nextKeyId: "next-2",
  nextPublicKey: nextKey2.publicKey,
};
const transition2 = await signer.signV2KeyTransition(
  transitionInput2,
  current2,
);
assert.ok(transition2.ok);
console.log(JSON.stringify({ ok: transition2.ok }));

{"ok":true}


## Verify the major 2 handover

Expected key identifiers and fingerprints come from the caller, independent of
the compact.

In [ ]:
const transitionFacts2 = verifier.v2.verifyKeyTransition(
  transition2.value.keyTransition,
  currentKey2,
  nextKey2,
  {
    transitionId: "transition-2",
    chainId: "audit:approvals",
    effectiveAt: 1_500,
    currentKeyId: "current-2",
    nextKeyId: "next-2",
    currentKeyFingerprint: currentFingerprint2,
    nextKeyFingerprint: nextFingerprint2,
  },
);
assert.ok(transitionFacts2.ok);
console.log(
  JSON.stringify({
    ok: transitionFacts2.ok,
    verification: transitionFacts2.value.verification,
  }),
);

{"ok":true,"verification":"authenticated_transition"}


## Sign a major 3 checkpoint

The current key signs the chain hash at a known sequence.


In [ ]:
const current3 = await createEs256Handle("current-3");
const next3 = await createEs256Handle("next-3");
const currentKey3 = {
  keyId: "current-3",
  publicKey: await current3.publicKey(),
  validFrom: 1_000,
  validBefore: 2_000,
};
const nextKey3 = {
  keyId: "next-3",
  publicKey: await next3.publicKey(),
  validFrom: 1_500,
  validBefore: null,
};
const currentFingerprint3 = verifier.base64urlDecode(
  verifier.strUtf8(await current3.thumbprint()),
);
const nextFingerprint3 = verifier.base64urlDecode(
  verifier.strUtf8(await next3.thumbprint()),
);
const checkpoint3 = {
  anchorId: "anchor-3",
  chainId: "audit:approvals",
  sequence: 0,
  chainHash: new Uint8Array(32),
};
const anchor3 = await signer.signV3Anchor(checkpoint3, current3, {
  anchoredAt: 1_400,
});
assert.ok(anchor3.ok);
console.log(JSON.stringify({ ok: anchor3.ok }));

{"ok":true}


## Verify the major 3 checkpoint

The caller supplies trusted key windows and the exact checkpoint fingerprint.


In [ ]:
const expectedAnchor3 = {
  ...checkpoint3,
  keyId: "current-3",
  anchoredAt: 1_400,
  keyFingerprint: currentFingerprint3,
};
const anchorFacts3 = verifier.v3.verifyHistoricalAnchor(
  anchor3.value.anchor,
  currentKey3,
  expectedAnchor3,
);
assert.ok(anchorFacts3.ok);
console.log(
  JSON.stringify({
    ok: anchorFacts3.ok,
    verification: anchorFacts3.value.verification,
    trust: anchorFacts3.value.trust,
  }),
);

{"ok":true,"verification":"signature_and_window","trust":"not_evaluated"}


## Refuse a changed major 3 checkpoint

A different chain hash must not verify under the original compact.


In [ ]:
const wrongHash3 = verifier.v3.verifyHistoricalAnchor(
  anchor3.value.anchor,
  currentKey3,
  { ...expectedAnchor3, chainHash: new Uint8Array(32).fill(1) },
);
assert.deepEqual(wrongHash3, { ok: false });
console.log(JSON.stringify(wrongHash3));

{"ok":false}


## Sign a major 3 key transition

Only the current key signs the handover to the next public key.


In [ ]:
const transitionInput3 = {
  transitionId: "transition-3",
  chainId: "audit:approvals",
  effectiveAt: 1_500,
  nextKeyId: "next-3",
  nextPublicKey: nextKey3.publicKey,
};
const transition3 = await signer.signV3KeyTransition(
  transitionInput3,
  current3,
);
assert.ok(transition3.ok);
console.log(JSON.stringify({ ok: transition3.ok }));

{"ok":true}


## Verify the major 3 handover

Expected key identifiers and fingerprints come from the caller, independent of
the compact.

In [ ]:
const transitionFacts3 = verifier.v3.verifyKeyTransition(
  transition3.value.keyTransition,
  currentKey3,
  nextKey3,
  {
    transitionId: "transition-3",
    chainId: "audit:approvals",
    effectiveAt: 1_500,
    currentKeyId: "current-3",
    nextKeyId: "next-3",
    currentKeyFingerprint: currentFingerprint3,
    nextKeyFingerprint: nextFingerprint3,
  },
);
assert.ok(transitionFacts3.ok);
console.log(
  JSON.stringify({
    ok: transitionFacts3.ok,
    verification: transitionFacts3.value.verification,
  }),
);

{"ok":true,"verification":"authenticated_transition"}


## Finish

Shut down the kernel to discard the demonstration keys. No private material was
persisted. Continue with the
[getting started guide](../guides/getting-started.md).